# 05c · Timestral-209M, LoRA fine-tuned, full replay on the 2026-10-25 cut

**What this notebook does (one paragraph):** it takes the pretrained forecasting model Timestral-209M and
*teaches it our electricity data* a little (this is called **fine-tuning**; we only train a small add-on called a
**LoRA adapter**, not all 209 million numbers inside the model). It tries four pre-declared training settings,
keeps the best one according to a score on a "tuning" period, trains that setting again for the "test" period and
for the final "replay" period, and saves its forecasts in the same file layout as notebooks 05 (Chronos-2) and 05b
(Timestral zero-shot). Notebook 06 can then put all models side by side.

**Status: EXPLORATORY.** The Timestral repository ships *no* training code, so the training loop in this
notebook is ours. It is built around the model's own `predict_block` function, the same function that produces the
zero-shot forecasts. Decided 2026-11-24 (fine-tune only if zero-shot is near or better than zero-shot Chronos-2),
approved 2026-11-26.

**How to read this notebook:** every code cell starts with a box of comments with four lines:
`WHAT` (what the cell does), `NEEDS` (what it reads from earlier cells), `MAKES` (what it creates or saves),
`CONDITIONS` (when it stops on purpose). Inside the cells, each variable has a comment explaining what it is.

## Glossary (read this first if a word is new)
| word | meaning |
|---|---|
| **device** | one electricity meter (one household or site). There are 13 in the study. |
| **target** | the series we forecast: `grid_import` (energy taken from the grid) or `grid_export` (energy fed into the grid, only for devices with solar panels). |
| **pair** | one (device, target) combination, e.g. "device X, grid_import". There are 23 pairs. Each pair is one time series. |
| **context** | the past values the model is allowed to read before a forecast: here the last 2048 hours (about 85 days). |
| **horizon** | how far ahead we forecast: 168 hours = 1 week. |
| **origin** | the moment a forecast is issued. It has seen every value *before* the origin and predicts the 168 hours starting at the origin. |
| **lead** | how many hours ahead one forecasted value is (lead 1 = first hour after the origin, lead 168 = last). |
| **window** | one (pair, origin) example: its context plus its forecast week. Training and scoring both work on windows. |
| **patch** | the model does not read single hours: it cuts the series into chunks of 32 hours, "patches", like words for a language model. |
| **covariate** | an extra series that helps predict the target: hour of day, weekday, holiday flag, weather (8 in total). We know them in advance, also for the future week. |
| **quantile** | "the value that the actual stays below with probability q". The model outputs 21 of them per hour (0.01 ... 0.99). The median (q = 0.5) is our single best guess ("point forecast"). |
| **pinball loss** | the training error for quantiles: it punishes a too-low 0.9-quantile much more than a too-high one, which pushes each quantile to its own level. |
| **zero-shot** | using the pretrained model as it is, with no training on our data. |
| **fine-tuning / LoRA** | training the model a bit on our data. LoRA adds small low-rank matrices to the model's layers and trains only those (a few million numbers). |
| **step** | one update of the adapter: take a batch of 16 random windows, measure the error, adjust the numbers slightly. |
| **learning rate (lr)** | how big each adjustment is. Too big = unstable, too small = learns nothing. |
| **block: tune / test / replay** | three consecutive time periods. *tune*: 2026-06-01 to 08-01 (we choose the training setting here). *test*: 08-01 to 09-01 (we score here). *replay*: from 09-01 on, the model is frozen at 09-01 and then forecasts "live". |
| **fold** | a slice of a block (14 days in tune, 7 days in test) so that scores are averaged over several periods. |
| **MAE** | mean absolute error, in kWh per hour. Lower is better. |

## Pre-declared choices (written BEFORE any run, never changed after seeing results)
| choice | value | why |
|---|---|---|
| model | `mistralai/Timestral-209M` | the primary size of the study |
| adapter | LoRA r=8, alpha=16 on `wQKV`, `wO`, `w12`, `w3` (all attention and MLP linear layers); the base weights stay frozen | the same treatment Chronos-2 got (LoRA r=8, alpha=16); updating all 209M weights would likely overfit 13 devices |
| loss | pinball over all 21 model quantiles, divided per window by the mean absolute observed context value (floor 0.01) | scale-free, so a big device does not dominate; the same idea as the model's own pretraining normalisation |
| optimiser | AdamW, weight decay 0.01, gradient clipping 1.0, 10% linear warm-up then cosine decay down to 10% of the peak learning rate | standard, fixed in advance |
| batch | 16 windows per step (each window = 1 target row + 8 known-covariate rows), processed as 4 chunks of 4 with summed gradients | the same update as a batch of 16; fits the GX10's memory cap |
| grid (steps, lr) | (250, 1e-4), (500, 1e-4), (1000, 1e-4), (1000, 3e-5) | same shape as Chronos-2's four-setting grid, with a short run to see where training stops helping |
| selection | the lowest tuning score (section 8), computed on the tuning period only; test and replay never choose anything | no peeking |
| refits | the best setting is trained again on labels before 2026-08-01 (for test) and before 2026-09-01 (for replay, then frozen) | same as notebook 05 |
| context / horizon | 2048 h / 168 h | same as the zero-shot run and every other model |
| covariates | the same 8 as notebooks 05 and 05b | fairness |
| output | quantiles 0.1, 0.25, 0.5, 0.75, 0.9; point forecast = median; clip at 0 and sort; **no** recalibration | same as the other models |
| training windows | an origin at every hour of the training period; every label used lies strictly before the cut; a window needs >=24 observed context hours and >=24 observed target hours; windows are drawn at random | no leakage |
| seed | 42 for every fit | reproducible |

**Known differences between training and forecasting (declared, not hidden):** training uses one forward pass per
window; the normal `forecast()` function averages that pass with a mirrored pass (the same input with the sign
flipped) and clips at 0. We keep that averaging unchanged for the fine-tuned model. Training also does not apply the
"real-value cap" that `forecast()` applies to extreme outputs.

**Weather caveat (true for all models):** the weather covariates are the *observed* weather at the forecast hour
("perfect forecast"), so every score here is an upper bound.

## 1. Parameters
**What this section is:** the one place where you can change settings. Papermill (the tool that runs the notebook on
the GX10) overrides these values from the command line, which is why the cell is tagged `parameters`.
**Rule:** change nothing here after you have looked at results. `DEVICES` and `BLOCKS` exist for testing only.

In [ ]:
# =====================================================================================================
# WHAT        Defines every setting of the notebook. Nothing is computed here.
# NEEDS       nothing.
# MAKES       the UPPER_CASE variables below, and PARAM_NAMES (the list of their names, saved in run_info.json).
# CONDITIONS  none here; later cells check the values (for example that FT_GRID is not empty).
# =====================================================================================================

# ---- which model and where the results go --------------------------------------------------------
MODEL_ID = "mistralai/Timestral-209M"       # the pretrained model on Hugging Face, 209 million numbers inside
RUN_NAME = "timestral-209M-finetuned-cov"   # name of the output folder AND the model name written in the
                                            # result files (column "model"); notebook 06 reads it from there

# ---- what to forecast -----------------------------------------------------------------------------
TARGETS = ["grid_export", "grid_import"]    # the two series; grid_export is only used for devices with solar PV
DEVICES = None                              # None means all 13 devices; or a list of device ids
                                            # (used for quick smoke tests; never for the real run)
BLOCKS = ["tune", "test", "replay"]         # which periods to run. A test run can use ["tune"] only.
HORIZON = 168                               # hours to forecast from each origin: 168 hours is one week
CONTEXT = 2048                              # hours of history the model may read: 2048 hours is 64 patches of 32 hours
MIN_CONTEXT_OBS = 24                        # a window is skipped if its context has fewer than 24 real (non-missing)
                                            # values; the same rule as notebooks 05 and 05b
LOCAL_TZ = "Europe/Rome"                    # calendar features (hour of day, weekday) and origins use LOCAL time

# ---- the three periods (local dates) ---------------------------------------------------------------
# tune   = [TUNE_START, TEST_START)  -> we choose the training setting here
# test   = [TEST_START, CUTOFF)      -> we score the chosen setting here
# replay = from CUTOFF onwards       -> the model is frozen at CUTOFF and forecasts "live"
TUNE_START = "2026-06-01"
TEST_START = "2026-08-01"
CUTOFF = "2026-09-01"
TUNE_FOLD_DAYS = 14                         # the tune period is cut into folds of 14 days
TEST_FOLD_DAYS = 7                          # the test period is cut into folds of 7 days
ORIGIN_STRIDE_H = 24                        # tune/test: one origin per day, at local midnight
REPLAY_STRIDE_H = 6                         # replay: one origin every 6 h (local 00:00, 06:00, 12:00, 18:00)
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]     # the quantile levels we save and score; the notebook needs these
                                            # five (median + 50% and 80% intervals)

# ---- LoRA (the small trainable add-on) -------------------------------------------------------------
LORA_R = 8                                  # "rank": the add-on is built from matrices with 8 inner columns.
                                            # Bigger = more capacity to change the model (and to overfit)
LORA_ALPHA = 16                             # scale of the add-on: its effect is multiplied by alpha divided by r, here 2
LORA_TARGETS = ["wQKV", "wO", "w12", "w3"]  # names of the layers that get an add-on: attention (query-key-value,
                                            # output) and the two feed-forward layers
# ---- the four declared training settings: (number of steps, peak learning rate) --------------------
FT_GRID = [(250, 1e-4), (500, 1e-4), (1000, 1e-4), (1000, 3e-5)]

# ---- training details (all fixed in advance) -------------------------------------------------------
FT_BATCH_WINDOWS = 16                       # windows used in ONE training step
FT_MICRO_BATCH_WINDOWS = 4                  # the 16 windows are processed in chunks of 4 and the gradients are
                                            # summed: the same update as one batch of 16, with a quarter of the
                                            # memory (a batch of 16 did not fit in the GPU memory cap)
WARMUP_FRAC = 0.1                           # the first 10% of the steps raise the learning rate from ~0 to its peak
MIN_LR_FRAC = 0.1                           # afterwards it decays (cosine) down to 10% of the peak
WEIGHT_DECAY = 0.01                         # small pull of the add-on weights toward 0 (against overfitting)
GRAD_CLIP = 1.0                             # an update is never larger than this norm (stability)
LOSS_SCALE_FLOOR = 0.01                     # the loss of a window is divided by its mean |value|, but never by less
                                            # than 0.01 kWh/h (avoids dividing by ~0 for an almost-empty series)
TRAIN_MIN_HORIZON_OBS = 24                  # a training window needs at least 24 real values in its target week

# ---- computing ---------------------------------------------------------------------------------------
EVAL_WINDOWS_PER_BATCH = 16                 # windows forecast per GPU pass (only affects speed and memory)
GPU_MEMORY_FRACTION = 0.16                  # this job may use at most 16% of the GX10's ~121 GB unified memory,
                                            # about 19 GB (the smoke test peaked at 13.5 GB; the chat model on the
                                            # same box holds ~52 GB; the box freezes if memory runs out)
SAVE_MERGED = True                          # True: also save the final weights as a folder that
                                            # TimestralForecaster.from_pretrained(folder) can load later
SEED = 42                                   # random seed: the same value for every fit, so runs are reproducible

# Taken once per kernel: this line collects the names of all UPPER_CASE variables above (so they can be written to
# run_info.json). Re-running this cell keeps the first list.
PARAM_NAMES = globals().get("PARAM_NAMES") or [k for k in dict(globals()) if k.isupper()]

## 2. Environment, hardware and data integrity
**What happens here:** we import the libraries, choose the compute device, and check that the data files are exactly
the frozen 2026-10-25 files.

- **GPU vs CPU.** A GPU does the many small multiplications inside a neural network in parallel, so it is much faster.
  On the GX10 the GPU and the CPU share one memory pool; `GPU_MEMORY_FRACTION` limits what this job may take.
- **SHA-256 fingerprints.** Every frozen file has a fingerprint (a long number computed from its bytes) recorded in
  `data/SHA256SUMS`. Change one byte and the fingerprint changes completely. If ours equals the recorded one, the file is
  exactly the frozen data and the comparison with the other models is valid; if not, we stop on purpose.

In [ ]:
# =====================================================================================================
# WHAT        Imports, picks GPU/CPU, limits GPU memory, creates the output folders, verifies the data.
# NEEDS       the parameters (RUN_NAME, GPU_MEMORY_FRACTION, FT_GRID, QUANTILES).
# MAKES       PARAMS (all parameters as a dict, for run_info.json), FT_GRID as a list of dicts,
#             DEVICE ("cuda" or "cpu"), NB_DIR (this notebook's folder), DATA_DIR, OUT_DIR (all outputs).
# CONDITIONS  stops if FT_GRID is empty, if a needed quantile level is missing, or if a data file's
#             fingerprint differs from data/SHA256SUMS.
# =====================================================================================================
import gc
import hashlib
import json
import math
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import timestral
from timestral import TimestralForecaster
from timestral.loading import resolve_checkpoint
from timestral.model import ROLE_TARGET

PARAMS = {k: globals()[k] for k in PARAM_NAMES}   # every parameter and its value, saved with the results
# FT_GRID came in as a list of (steps, lr) pairs; turn each into a small dict so the code reads FT_GRID[i]["steps"]
FT_GRID = [dict(steps=int(s), lr=float(lr)) for s, lr in FT_GRID]
assert FT_GRID, "FT_GRID is empty"
assert {0.1, 0.25, 0.5, 0.75, 0.9} <= set(QUANTILES), "the median and the 50%/80% intervals need these levels"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # use the GPU when there is one
if DEVICE == "cuda":
    torch.cuda.set_per_process_memory_fraction(GPU_MEMORY_FRACTION)  # hard cap: this process cannot take more
    torch.cuda.reset_peak_memory_stats()                              # so that we can report the peak at the end
print("torch", torch.__version__, "| device:", DEVICE, "| timestral:", Path(timestral.__file__).parent)

NB_DIR = Path.cwd().resolve()           # the folder the notebook runs in (on the GX10: the job folder)
DATA_DIR = NB_DIR / "data"              # the frozen data lives in a "data" folder next to the notebook
OUT_DIR = NB_DIR / "out" / RUN_NAME     # everything we produce goes here
for sub in ("cache", "adapters", "models"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
# cache/    : forecasts saved per stage, so a crashed run resumes instead of starting over
# adapters/ : the trained LoRA numbers of every fit (small files)
# models/   : the final test and replay models as full folders (large, ~0.8 GB each)


def sha256(path):
    """Fingerprint of a file, read in 1 MB pieces so memory stays small."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# Each line of SHA256SUMS is "<fingerprint>  <filename>"; build {filename: fingerprint}.
recorded = dict(reversed(line.split()) for line in (DATA_DIR / "SHA256SUMS").read_text().splitlines()
                if line.strip())
for name in ["processed_hourly_2026-10-25.parquet", "device_quality_2026-10-25.csv"]:
    ok = sha256(DATA_DIR / name) == recorded[name]
    print("OK      " if ok else "MISMATCH", name)
    assert ok, f"{name} differs from the frozen cut: stop"
print("output:", OUT_DIR)

## 3. Data, covariates and one hourly grid
**What happens here:**
1. `processed_hourly` has one row per device per hour (UTC). For each device we build a series with *no missing
   hours*: hours without a measurement become `NaN` ("not a number" = unknown), so "the 2048 hours before an origin" is
   just an array slice.
2. `HAS_PV` marks devices with solar panels. Only they can export energy, so `grid_export` is forecast only for them.
3. We build the 8 covariates for *every hour*, past and future, on one common hourly UTC grid called `HOURS`:

| covariate | idea |
|---|---|
| `hour_sin`, `hour_cos` | hour of day as a point on a circle (so 23:00 and 00:00 are neighbours) |
| `dow_sin`, `dow_cos` | day of week as a point on a circle |
| `is_holiday` | 1 on Italian public holidays |
| `global_tilted_irradiance`, `temperature_2m`, `cloud_cover` | observed weather at the site (the same for every device) |

Why sin and cos? Hour 23 and hour 0 are neighbours, but the numbers 23 and 0 are far apart. Mapping the hour to an
angle `2*pi*hour/24` and using `(sin, cos)` puts them next to each other on a circle; two numbers are needed because
sin alone cannot tell 6:00 from 18:00.

In [ ]:
# =====================================================================================================
# WHAT        Loads the frozen data and builds, for every (device, target) pair, an hourly series Y, plus the 8
#             covariates COV_ARR on one common hourly grid HOURS.
# NEEDS       DATA_DIR, DEVICES, TARGETS, CONTEXT, HORIZON, LOCAL_TZ.
# MAKES       DEV   {device: hourly table},          PAIRS  list of (device, target),
#             Y     {(device, target): array},        COV_ARR array (hours x 8),
#             CS    {(device, target): running count of real values} (a fast "how many real values in a slice"),
#             HOURS the common hourly grid,          pos(ts) position of an hour on that grid.
# CONDITIONS  none (an empty selection would simply give zero pairs and fail later).
# =====================================================================================================
PT = pd.read_parquet(DATA_DIR / "processed_hourly_2026-10-25.parquet")   # one row per device per hour
quality = pd.read_csv(DATA_DIR / "device_quality_2026-10-25.csv").set_index("device_id")
HAS_PV = quality["has_pv"].astype(bool).to_dict()          # {device: True if it has solar panels}
ALL_DEVICES = sorted(DEVICES or PT["device_id"].unique())  # DEVICES=None means every device in the data
PT = PT[PT["device_id"].isin(ALL_DEVICES)]
# {device: hourly table}. drop_duplicates keeps the last row if an hour appears twice; asfreq("h") inserts the
# missing hours as NaN rows.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}
# every (device, target) combination we forecast: grid_export only where the device has PV
PAIRS = [(d, t) for t in TARGETS for d in ALL_DEVICES if t != "grid_export" or HAS_PV.get(d, False)]
PAIR_ID = {p: i for i, p in enumerate(PAIRS)}               # a number per pair, for fast random sampling

# Italian public holidays (copied unchanged from notebooks 05 and 05b).
HOLIDAYS = {pd.Timestamp(d).date() for d in [
    "2025-01-01", "2025-01-06", "2025-04-21", "2025-04-25", "2025-05-01", "2025-06-02",
    "2025-08-15", "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-06", "2026-04-25", "2026-05-01", "2026-06-02",
    "2026-08-15", "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26"]}
WEATHER_COLS = ["global_tilted_irradiance", "temperature_2m", "cloud_cover"]

PATH_H = pd.Timedelta(hours=HORIZON)                        # the forecast length as a time span (168 h)
# ONE common hourly UTC grid. It starts CONTEXT hours before the first data point (so the first origin has a full
# context) and ends HORIZON hours after the last (so covariates exist for the whole forecast week).
HOURS = pd.date_range(PT["ts_hour"].min() - pd.Timedelta(hours=CONTEXT),
                      PT["ts_hour"].max() + PATH_H, freq="h")
local = HOURS.tz_convert(LOCAL_TZ)                          # calendar features follow LOCAL clock time, not UTC
COV = pd.DataFrame({"hour_sin": np.sin(2 * np.pi * local.hour / 24),
                    "hour_cos": np.cos(2 * np.pi * local.hour / 24),
                    "dow_sin": np.sin(2 * np.pi * local.day_of_week / 7),
                    "dow_cos": np.cos(2 * np.pi * local.day_of_week / 7),
                    "is_holiday": pd.Index(local.date).isin(HOLIDAYS).astype(float)}, index=HOURS)
# One site, so the weather is the same for every device: take the first row per hour and interpolate small gaps.
COV[WEATHER_COLS] = (PT.groupby("ts_hour")[WEATHER_COLS].first().reindex(HOURS)
                     .interpolate(limit_direction="both"))
COV_ARR = COV.to_numpy(np.float32)                          # the 8 covariates as an array: (hours, 8)
# Y[(device, target)] is the hourly series laid on the grid HOURS (NaN = unknown)
Y = {(d, t): DEV[d][t].reindex(HOURS).to_numpy(np.float32) for d, t in PAIRS}
# CS[pair][i] = how many real values the series has BEFORE position i. Then "real values in slice a:b" is
# CS[b] - CS[a], one subtraction instead of a loop.
CS = {pair: np.concatenate([[0], np.cumsum(np.isfinite(y))]) for pair, y in Y.items()}


def pos(ts):
    """Position (0, 1, 2, ...) of a timestamp on the common hourly grid HOURS."""
    return int((ts - HOURS[0]) / pd.Timedelta(hours=1))


print(len(ALL_DEVICES), "devices ->", len(PAIRS), "device/target pairs |", COV.shape[1], "covariates |",
      len(HOURS), "grid hours")

## 4. Origins, blocks and the scored windows
**Vocabulary.** An **origin** is the moment a forecast is issued. A forecast from origin `o` has seen every label
*before* `o` and predicts hours `o, o+1, ..., o+167` (lead 1..168). Nothing at or after `o` is ever shown to the
model as target history (no look-ahead).

**Which origins? (same rule as notebook 05)**
- **tune** origins: every local midnight in `[TUNE_START, TEST_START)`, in 14-day folds, and the whole 168 h week must
  still lie inside the block;
- **test** origins: every local midnight in `[TEST_START, CUTOFF)`, in 7-day folds, same week rule;
- **replay** origins: local 00:00, 06:00, 12:00, 18:00 from `CUTOFF` on, as long as the full 168 h of actuals exist.

A window is also skipped when its context has fewer than 24 real values (same rule as notebooks 05 and 05b).

In [ ]:
# =====================================================================================================
# WHAT        Lists every window (pair x origin) we will forecast and score, and defines two helpers:
#             batch_inputs (cut the context/covariates/target week out of the grid) and expand (one row per
#             window x lead, with the actual value).
# NEEDS       Y, COV_ARR, CS, HOURS, PAIRS and the period parameters.
# MAKES       WINS  table with one row per window: block, fold, device_id, target, origin, p (grid position),
#             ctx_obs (real values in its context);   T_TUNE, T_TEST, CUT (period starts, UTC);
#             QCOLS (names of the quantile columns), KEY (columns that identify one forecast row).
# CONDITIONS  windows with fewer than MIN_CONTEXT_OBS real context values are dropped; only the blocks listed
#             in BLOCKS are kept.
# =====================================================================================================
# The period boundaries as local-time timestamps ("cut" is the start of the replay).
L = {k: pd.Timestamp(v, tz=LOCAL_TZ) for k, v in dict(tune=TUNE_START, test=TEST_START, cut=CUTOFF).items()}
rows = []   # (block, fold, origin) for the tune and test periods
for block, start, end, days in [("tune", L["tune"], L["test"], TUNE_FOLD_DAYS),
                                ("test", L["test"], L["cut"], TEST_FOLD_DAYS)]:
    for fold in pd.date_range(start, end, freq=f"{days}D", inclusive="left"):       # fold start days
        stop = min(fold + pd.Timedelta(days=days), end)                              # fold end (or block end)
        rows += [(block, fold, o) for o in pd.date_range(fold, stop, freq=f"{ORIGIN_STRIDE_H}h",
                                                         inclusive="left")
                 if o + PATH_H <= end]                                               # the full week must fit
BLK = pd.DataFrame(rows, columns=["block", "fold", "origin"])
BLK[["fold", "origin"]] = BLK[["fold", "origin"]].apply(lambda c: c.dt.tz_convert("UTC"))   # all times in UTC
T_TUNE, T_TEST, CUT = (L[k].tz_convert("UTC") for k in ("tune", "test", "cut"))

parts = []
for d, t in PAIRS:
    # last replay origin: the one whose full week of actuals is still inside the data
    last = DEV[d].index[-1] - PATH_H + pd.Timedelta(hours=1)
    rep = pd.date_range(CUT, last, freq=f"{REPLAY_STRIDE_H}h")
    o = pd.concat([BLK, pd.DataFrame({"block": "replay", "fold": CUT, "origin": rep})], ignore_index=True)
    o["device_id"], o["target"] = d, t
    parts.append(o)
WINS = pd.concat(parts, ignore_index=True)
WINS["p"] = ((WINS["origin"] - HOURS[0]) // pd.Timedelta(hours=1)).astype(int)   # position of the origin on HOURS
# how many real values the 2048 h before each origin contain
WINS["ctx_obs"] = [CS[(d, t)][p] - CS[(d, t)][p - CONTEXT]
                   for d, t, p in zip(WINS["device_id"], WINS["target"], WINS["p"])]
WINS = WINS[(WINS["ctx_obs"] >= MIN_CONTEXT_OBS) & WINS["block"].isin(BLOCKS)].reset_index(drop=True)
print("windows per block:")
print(WINS.groupby("block").size().to_string())

QCOLS = [f"q{l}" for l in QUANTILES]                 # "q0.1", "q0.25", ... : names of the quantile columns
KEY = ["device_id", "target", "origin", "horizon"]   # the four columns that identify one forecast value


def batch_inputs(pairs, ps):
    """Cut the model inputs of several windows out of the grid.

    pairs: list of (device, target); ps: list of origin positions on HOURS (same length).
    Returns three arrays: ctx (B, 2048) the context values (NaN = missing),
    cov (B, 8, 2048+168) the 8 covariates over context AND forecast week, fut (B, 168) the actual target week
    (used only as the training label; never given to the model as input).
    """
    ctx = np.stack([Y[pr][p - CONTEXT:p] for pr, p in zip(pairs, ps)]).astype(np.float32)           # before the origin
    cov = np.stack([COV_ARR[p - CONTEXT:p + HORIZON].T for p in ps]).astype(np.float32)             # covariates
    fut = np.stack([Y[pr][p:p + HORIZON] for pr, p in zip(pairs, ps)]).astype(np.float32)           # the label
    return ctx, cov, fut


def expand(W):
    """A table with one row per window x lead (1..168) for the windows in W, keeping W's order.

    Columns: block, fold, device_id, target, origin, horizon (the lead), ts_hour (the hour being forecast) and
    actual (the true value, NaN where the meter had a gap).
    """
    n = len(W)
    h = np.tile(np.arange(1, HORIZON + 1), n)        # 1..168 repeated for every window
    df = pd.DataFrame({"block": np.repeat(W["block"].to_numpy(), HORIZON),
                       "fold": np.repeat(W["fold"].to_numpy(), HORIZON),
                       "device_id": np.repeat(W["device_id"].to_numpy(), HORIZON),
                       "target": np.repeat(W["target"].to_numpy(), HORIZON),
                       "origin": np.repeat(W["origin"].to_numpy(), HORIZON),
                       "horizon": h})
    df["ts_hour"] = df["origin"] + pd.to_timedelta(h - 1, unit="h")     # origin + (lead - 1) hours
    df["actual"] = np.concatenate([Y[(d, t)][p:p + HORIZON] for d, t, p in
                                   zip(W["device_id"], W["target"], W["p"])]).astype(np.float64)
    return df

## 5. The model, the LoRA adapter and one differentiable forecast
**LoRA in one sentence.** Each big weight matrix `W` of the model stays frozen; we add a small correction built from
two thin matrices `A` (`r x in`) and `B` (`out x r`): the layer computes `W x + (alpha / r) * B (A x)`. `B` starts at
zero, so *before any training the model is exactly the pretrained one*. Only `A` and `B` are trained (a few million
numbers instead of 209 million).

**Training forward pass (`raw_quantiles`).** It builds the model input exactly like the normal `forecast()` does (one
target row plus 8 known-covariate rows per window), calls the model's own `predict_block` for the whole 168 h at once
(6 patches of 32 h; the model can decode up to 32 patches in one block, so 168 h is a single block), applies the same
"refine" step, and returns the raw quantiles `(windows, 168 leads, 21 levels)` **with gradients** (the information that
lets training know in which direction to change the numbers). Section 6 proves that it matches the normal forecast.

**Loss (`pinball_loss`).** For a quantile level `tau`, forecast `q` and actual `y`: `max(tau*(y-q), (tau-1)*(y-q))`. If
the actual is above the quantile we pay `tau` per unit of miss, if below `1-tau`. For `tau = 0.9`, being too low costs
9 times more than being too high, which pushes `q` to the value that 90% of actuals fall under. We average over all 21
levels and the 168 leads (skipping hours with no actual), divide each window by its own typical size (mean absolute
context value, never less than `LOSS_SCALE_FLOOR`), and average over the windows.

In [ ]:
# =====================================================================================================
# WHAT        Defines the building blocks of fine-tuning: loading the model, the LoRA layer, how to attach LoRA to
#             the model, the differentiable forecast, and the loss. (Definitions only: nothing runs yet.)
# NEEDS       MODEL_ID, DEVICE, LORA_*, HORIZON, CONTEXT, LOSS_SCALE_FLOOR.
# MAKES       load_forecaster(), LoRALinear, add_lora(), raw_quantiles(), pinball_loss(); PS (patch size).
# CONDITIONS  load_forecaster() stops if CONTEXT is not a multiple of the patch size, or if the horizon would need
#             more than one decoding block (then training on one block would not match forecasting).
# =====================================================================================================
PS = None  # patch size (32); set when the first model is loaded


def load_forecaster():
    """Load the pretrained Timestral model (downloaded once, then read from the disk cache)."""
    global PS
    f = TimestralForecaster.from_pretrained(MODEL_ID, device=DEVICE)
    PS = f.patch_size                                                           # 32 hours per patch
    assert CONTEXT % PS == 0, "the context must be a whole number of patches"
    assert math.ceil(HORIZON / PS) <= f.per_shot_horizon_patches, "the horizon must fit in one decoding block"
    return f


class LoRALinear(nn.Module):
    """A frozen linear layer plus a small trainable low-rank correction (see the text above)."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base                  # the original layer, frozen
        self.scaling = alpha / r          # how strongly the correction counts (16 / 8 = 2)
        dev = base.weight.device
        # A starts random (small), B starts at zero: so the correction is zero before training.
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scaling

    def merged_weight(self):
        """The original weight with the correction folded in: used to save a plain model folder."""
        return self.base.weight + self.scaling * (self.B @ self.A)


def add_lora(model):
    """Freeze every weight of `model`, then wrap each layer named in LORA_TARGETS with a LoRALinear.

    Returns how many layers were wrapped. Only the new A and B matrices are trainable afterwards.
    """
    for p in model.parameters():
        p.requires_grad_(False)                                    # frozen: training will not change these
    n = 0
    for mod in list(model.modules()):                              # list() first: we change the model while walking
        for child_name, child in list(mod.named_children()):
            if child_name in LORA_TARGETS and isinstance(child, nn.Linear):
                setattr(mod, child_name, LoRALinear(child, LORA_R, LORA_ALPHA))
                n += 1
    for p in model.parameters():
        if p.dtype != torch.float32:
            raise TypeError("expected float32 weights")           # inference was validated in float32 only
    return n


def raw_quantiles(fc, ctx, cov):
    """Differentiable forecast: the training-time twin of fc.forecast().

    fc: a TimestralForecaster; ctx: (B, 2048) context; cov: (B, 8, 2048+168) covariates.
    Returns (B, 168, 21): for every window, lead and model quantile level, the forecast in kWh/h.
    """
    x = torch.as_tensor(ctx, dtype=torch.float32, device=fc.device)[:, None, :]   # (B, 1 variate, 2048)
    m = torch.isfinite(x)                                  # True where the value is real, False where it is NaN
    x = torch.where(m, x, torch.zeros_like(x))             # NaN -> 0 (the mask m tells the model they are missing)
    known = torch.as_tensor(cov, dtype=torch.float32, device=fc.device)
    # Stack the target row and the 8 covariate rows of every window (the model's own routine).
    rows, mask, role_ids, kf, kfm, n_tgt = fc._assemble_rows(x, m, HORIZON, None, known)
    b, v = rows.shape[0], rows.shape[1]                    # b windows, v = 1 target + 8 covariate rows each
    context, mask = rows.reshape(b * v, -1), mask.reshape(b * v, -1)
    gid = torch.arange(b, device=fc.device).repeat_interleave(v)   # which rows belong to the same window
    # ONE block of 6 patches (6 x 32 = 192 h >= 168 h)
    q = fc.model.predict_block(context, mask, gid, math.ceil(HORIZON / PS), role_ids, kf, kfm)
    q = fc._refine_block(q, context, mask, role_ids == ROLE_TARGET)    # the same refine step as in forecast()
    # keep the target row of every window (row 0) and the first 168 of the 192 hours
    return q.reshape(b, v, q.shape[1], q.shape[2])[:, 0, :HORIZON]


def pinball_loss(q, fut, ctx, levels):
    """The scale-normalised pinball loss (a single number) of a batch.

    q: (B, 168, 21) forecasts; fut: (B, 168) actuals (NaN = unknown, ignored); ctx: (B, 2048) the context
    (only used to measure each window's typical size); levels: the 21 quantile levels as a tensor.
    """
    fut = torch.as_tensor(fut, device=q.device)
    ctx = torch.as_tensor(ctx, device=q.device)
    m = torch.isfinite(fut)                                                       # hours that have an actual
    d = torch.where(m, fut, torch.zeros_like(fut))[..., None] - q                 # actual - forecast, per level
    l = torch.maximum(levels * d, (levels - 1) * d).mean(-1)                      # pinball, averaged over 21 levels
    per_window = (l * m).sum(1) / m.sum(1).clamp_min(1)                           # average over the known hours
    cm = torch.isfinite(ctx)
    scale = torch.where(cm, ctx.abs(), torch.zeros_like(ctx)).sum(1) / cm.sum(1).clamp_min(1)   # mean |value|
    return (per_window / scale.clamp_min(LOSS_SCALE_FLOOR)).mean()                # make it scale-free, average

## 6. Equivalence checks (they run before any training)
Fine-tuning code is easy to get subtly wrong, so three checks run first. If any fails, the notebook stops: nothing
trained after a failed check could be trusted.

1. **Same forward pass.** `raw_quantiles` (training) must reproduce the model's own inference pass `_forecast_raw`.
2. **LoRA starts as the identity.** With `B = 0`, the LoRA-wrapped model must give the same forecasts as the
   pretrained one.
3. **Gradients flow.** One backward pass must give a non-zero gradient on the LoRA `B` matrices, otherwise nothing
   would be learned.

"Relative difference" = the largest absolute difference between two forecasts divided by the largest forecast value.

In [ ]:
# =====================================================================================================
# WHAT        Runs the three equivalence checks above on 6 sample windows.
# NEEDS       WINS, batch_inputs, load_forecaster, add_lora, raw_quantiles, pinball_loss.
# MAKES       LEVELS (the model's 21 quantile levels as a tensor, used by the loss). Frees the temporary model.
# CONDITIONS  STOPS the notebook if: relative difference 1 >= 1e-3, or relative difference 2 >= 1e-5, or the
#             gradient is zero, or the loss is not a finite number.
# =====================================================================================================
sample = WINS.head(6)                                                    # 6 windows are enough for a check
s_pairs = list(zip(sample["device_id"], sample["target"]))
s_ctx, s_cov, s_fut = batch_inputs(s_pairs, sample["p"].tolist())

base = load_forecaster()
LEVELS = torch.tensor(base.config.quantile_knots, dtype=torch.float32, device=base.device)   # 0.01 ... 0.99
x = torch.as_tensor(s_ctx, device=base.device)[:, None, :]
xm = torch.isfinite(x)
with torch.no_grad():                                                    # no gradients needed for checks 1 and 2
    # check 1: the model's own inference pass vs our training-time pass
    ref = base._forecast_raw(torch.where(xm, x, torch.zeros_like(x)), xm, HORIZON,
                             known_covariates=torch.as_tensor(s_cov, device=base.device))[:, 0]
    mine = raw_quantiles(base, s_ctx, s_cov)
    rel1 = ((ref - mine).abs().max() / ref.abs().max().clamp_min(1e-9)).item()
n_wrapped = add_lora(base.model)                                         # attach the (still empty) adapters
with torch.no_grad():                                                    # check 2: untrained LoRA vs pretrained
    lora0 = raw_quantiles(base, s_ctx, s_cov)
    rel2 = ((mine - lora0).abs().max() / mine.abs().max().clamp_min(1e-9)).item()
base.model.train()                                                       # check 3: one forward + backward pass
loss = pinball_loss(raw_quantiles(base, s_ctx, s_cov), s_fut, s_ctx, LEVELS)
loss.backward()
# biggest gradient on any LoRA "B" matrix (A has zero gradient at the start because B is zero: that is expected)
gmax = max(p.grad.abs().max().item() for n_, p in base.model.named_parameters()
           if p.requires_grad and n_.endswith(".B") and p.grad is not None)
n_train = sum(p.numel() for p in base.model.parameters() if p.requires_grad)
print(f"1. training pass vs inference pass: max relative difference {rel1:.2e}")
print(f"2. untrained LoRA vs pretrained:    max relative difference {rel2:.2e}  ({n_wrapped} layers wrapped)")
print(f"3. loss {loss.item():.4f} | max |grad| on a LoRA B matrix {gmax:.2e} | trainable parameters {n_train / 1e6:.2f} M")
assert rel1 < 1e-3 and rel2 < 1e-5 and gmax > 0 and np.isfinite(loss.item()), "an equivalence check failed: stop"
del base, ref, mine, lora0, loss                                         # free the memory before real training
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 7. Training and forecasting
**Training windows.** For a fit whose training data must end at time `end`, every hourly origin `p` with
`p + 168 h <= end` is a candidate: its context *and* its target week lie entirely before `end`, so no label from the
period being evaluated can leak in (this is asserted in the code). A candidate also needs at least 24 real context
values and at least 24 real values in its target week. Each step draws 16 candidates uniformly at random.

**One training step:** take 16 random windows, in 4 chunks of 4 (to save memory) let the model forecast them
(`raw_quantiles`), measure the error (`pinball_loss`), compute in which direction each LoRA number should move to
reduce it (`loss.backward()`, summed over the chunks, which equals one batch of 16), cap the size of the move
(gradient clipping), and move (`opt.step()`). The learning rate follows a schedule: warm-up, then
cosine decay.

**Forecasting** (`forecast_windows`) uses the normal `forecast()` path (mirror average, clip at 0), the same as notebook
05b, then keeps our 5 quantile levels out of the model's 21, clips at 0 again and sorts the levels so they cannot
cross. The point forecast is the median.

**Resume support (`cached_forecast`):** the forecasts of every stage are saved to `out/<run>/cache/`. If the notebook
crashes (or is run again), finished stages are read from disk instead of being recomputed.

In [ ]:
# =====================================================================================================
# WHAT        Defines the training loop (finetune), the forecasting routine (forecast_windows) and the resume
#             helper (cached_forecast). Definitions only: nothing runs yet.
# NEEDS       everything from sections 2-6 (data, windows, LoRA, loss) and the training parameters.
# MAKES       LOSS_LOG (a list that collects the loss of every step), train_pool(), finetune(), level_index(),
#             forecast_windows(), cached_forecast().
# CONDITIONS  train_pool stops if there are no training windows or if one would reach past the cut (leakage guard).
# =====================================================================================================
LOSS_LOG = []   # one entry per training step: which fit, step number, loss, learning rate (saved as loss.csv)


def train_pool(end):
    """All possible training windows whose labels all lie strictly before `end`.

    Returns an array with one row per window: (pair number, origin position on HOURS).
    """
    pe = pos(end)                                           # position of the cut on the grid
    out = []
    for pair in PAIRS:
        cs = CS[pair]                                       # running count of real values of this series
        p = np.arange(CONTEXT, pe - HORIZON + 1)            # origins whose whole week ends at or before the cut
        ok = (((cs[p] - cs[p - CONTEXT]) >= MIN_CONTEXT_OBS)          # enough real values in the context
              & ((cs[p + HORIZON] - cs[p]) >= TRAIN_MIN_HORIZON_OBS)) # and in the target week
        out.append(np.stack([np.full(int(ok.sum()), PAIR_ID[pair]), p[ok]], axis=1))
    pool = np.concatenate(out)
    assert len(pool) > 0, "no training windows"
    assert (pool[:, 1] + HORIZON <= pe).all(), "a training window reaches past the cut"   # leakage guard
    return pool


def finetune(end, steps, lr, name):
    """Train a fresh LoRA adapter on the windows before `end`.

    end: the cut (labels at or after it are never used); steps: number of training steps; lr: peak learning rate;
    name: label for logs and saved files. Returns (the fine-tuned forecaster, seconds it took).
    """
    torch.manual_seed(SEED)                                 # same random start for every fit
    rng = np.random.default_rng(SEED)                       # random generator that picks the training windows
    fc = load_forecaster()                                  # a fresh pretrained model every time
    add_lora(fc.model)                                      # freeze it and attach the adapters
    params = [p for p in fc.model.parameters() if p.requires_grad]   # only the LoRA numbers
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    warm = max(1, int(WARMUP_FRAC * steps))                 # number of warm-up steps

    def lr_at(i):
        """Learning rate at step i: linear warm-up to `lr`, then cosine decay to MIN_LR_FRAC * lr."""
        if i < warm:
            return lr * (i + 1) / warm
        t = (i - warm) / max(1, steps - warm)               # 0 -> 1 over the decay phase
        return lr * (MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * t)))

    pool = train_pool(end)
    print(f"[{name}] {steps} steps, lr {lr:g}, {len(pool):,} candidate windows before {end:%Y-%m-%d %H:%M} UTC")
    fc.model.train()
    t0 = time.time()
    for i in range(steps):
        sel = pool[rng.integers(0, len(pool), FT_BATCH_WINDOWS)]                  # 16 random windows
        for g in opt.param_groups:
            g["lr"] = lr_at(i)                                                    # set this step's learning rate
        step_loss = 0.0                                                           # the loss of the whole step
        for j in range(0, len(sel), FT_MICRO_BATCH_WINDOWS):                      # chunks of 4 windows
            part = sel[j:j + FT_MICRO_BATCH_WINDOWS]
            ctx, cov, fut = batch_inputs([PAIRS[k] for k in part[:, 0]], part[:, 1].tolist())
            # weight each chunk by its share of the 16 windows, so the sum over chunks is the mean over all 16
            loss = pinball_loss(raw_quantiles(fc, ctx, cov), fut, ctx, LEVELS) * len(part) / len(sel)
            loss.backward()                                                       # which way to move each number
            step_loss += loss.item()                                              # (gradients of chunks are added up)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)                         # cap the size of the move
        opt.step()                                                                # move
        opt.zero_grad(set_to_none=True)                                           # forget this step's gradients
        LOSS_LOG.append(dict(name=name, step=i + 1, loss=step_loss, lr=lr_at(i)))
        if (i + 1) % 25 == 0 or i == 0:                                           # progress line every 25 steps
            last = np.mean([r["loss"] for r in LOSS_LOG[-25:]])
            print(f"  [{name}] step {i + 1}/{steps} loss(25) {last:.4f} | {time.time() - t0:.0f}s", flush=True)
    fc.model.eval()
    torch.save({n: p.detach().cpu() for n, p in fc.model.named_parameters() if p.requires_grad},
               OUT_DIR / "adapters" / f"{name}.pt")                               # save the trained LoRA numbers
    pd.DataFrame(LOSS_LOG).to_csv(OUT_DIR / "loss.csv", index=False)
    return fc, time.time() - t0


def level_index(levels):
    """Position of each of our QUANTILES inside the model's 21 output levels."""
    lv = np.asarray(levels, dtype=float)
    return [int(np.argmin(np.abs(lv - q))) for q in QUANTILES]


def forecast_windows(fc, W, label=""):
    """Forecast every window in the table `W` with the normal fc.forecast() path.

    fc: a (fine-tuned or pretrained) TimestralForecaster; W: rows of WINS; label: text for the progress lines.
    Returns a table with one row per window x lead: the identifying columns, the actual, the 5 quantile columns
    and `point` (= the median). Rows without an actual are dropped (a meter gap), as in notebook 05.
    """
    out, t0 = [], time.time()
    for i in range(0, len(W), EVAL_WINDOWS_PER_BATCH):
        b = W.iloc[i:i + EVAL_WINDOWS_PER_BATCH]                                  # a small batch of windows
        ctx, cov, _ = batch_inputs(list(zip(b["device_id"], b["target"])), b["p"].tolist())
        f = fc.forecast(ctx[:, None, :], prediction_length=HORIZON, known_covariates=cov)
        q = f.quantiles.float().cpu().numpy()[:, 0][:, :, level_index(f.quantile_levels)]   # keep our 5 levels
        out.append(np.sort(np.maximum(0.0, q), axis=-1))                          # clip at 0, then no crossing
        if (i // EVAL_WINDOWS_PER_BATCH) % 25 == 0:
            print(f"  {label} {i + len(b)}/{len(W)} windows | {time.time() - t0:.0f}s", flush=True)
    q = np.concatenate(out).reshape(-1, len(QUANTILES))                           # (windows x 168, 5)
    df = expand(W)                                                                # same row order as q
    df[QCOLS] = q
    df["point"] = df["q0.5"]                                                      # point forecast = median
    return df.dropna(subset=["actual"]).reset_index(drop=True)


def cached_forecast(name, make):
    """Return the saved forecasts `name` if they exist, otherwise call make(), save and return its result."""
    path = OUT_DIR / "cache" / f"{name}.parquet"
    if path.exists():
        print("cached:", name)
        return pd.read_parquet(path)
    df = make()
    df.to_parquet(path, index=False)
    return df

## 8. Tuning: which of the four settings is best?
**What happens here:** every setting in `FT_GRID` is trained once on the labels *before* `TUNE_START`, then forecasts
every tuning window (June and July 2026). The pretrained model (zero-shot) is scored the same way for reference.

**The tuning score (same as notebook 05):** for a series in a fold, take the MAE over leads 1-48, 1-96 and 1-168 and
average these three numbers (so near and far horizons count equally); a setting's score is the mean over folds of the
mean over series. **Lower is better. Only this score selects the setting** (`BEST`); the test and replay periods are
never used to choose anything.

In [ ]:
# =====================================================================================================
# WHAT        Trains and scores every setting of FT_GRID on the tuning period, plus the zero-shot model, and picks
#             the best setting.
# NEEDS       WINS (block "tune"), finetune, forecast_windows, cached_forecast, FT_GRID, T_TUNE.
# MAKES       GRID (table: one row per setting with its score), BEST (the winning setting), TUNE_SECONDS;
#             saves out/<run>/grid.csv and best_params.json, and one forecast file per setting in cache/.
# CONDITIONS  if "tune" is not in BLOCKS nothing is tuned and BEST stays None (later cells then skip refits).
# =====================================================================================================
LEAD_CUTS = [48, 96, HORIZON]    # the three horizons whose MAEs are averaged into the tuning score


def tune_score(df):
    """The tuning score of a forecast table (see the text above). Lower is better."""
    r = df.assign(err=(df["point"] - df["actual"]).abs())                          # absolute error per row
    per_series = sum(r[r["horizon"] <= c].groupby(["fold", "device_id", "target"])["err"].mean()
                     for c in LEAD_CUTS) / len(LEAD_CUTS)                          # mean of the three MAEs
    return float(per_series.groupby("fold").mean().mean())                         # mean over series, then folds


TUNE_W = WINS[WINS["block"] == "tune"]     # the windows of the tuning period
t0 = time.time()
GRID = []
if len(TUNE_W):
    # reference row: the pretrained model with no training
    zs = cached_forecast("zeroshot_tune", lambda: forecast_windows(load_forecaster(), TUNE_W, "zero-shot tune"))
    GRID.append(dict(setting="zeroshot", steps=0, lr=0.0, ft_seconds=0.0, score=tune_score(zs)))
    for i, g in enumerate(FT_GRID):                       # g = {"steps": ..., "lr": ...}
        name = f"tune{i}"
        secs = [0.0]                                       # training time of this setting (filled by make())

        def make(g=g, name=name):
            fc, secs[0] = finetune(T_TUNE, **g, name=name)         # train on labels before TUNE_START
            df = forecast_windows(fc, TUNE_W, name)                # forecast the tuning windows
            del fc
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()                           # give the memory back before the next fit
            return df

        df = cached_forecast(name, make)
        GRID.append(dict(setting=name, **g, ft_seconds=secs[0], score=tune_score(df)))
    GRID = pd.DataFrame(GRID).set_index("setting")
    GRID.to_csv(OUT_DIR / "grid.csv")
    # best = the fine-tuned setting (row 0 is zero-shot, so skip it) with the lowest score
    BEST = FT_GRID[int(GRID["score"].iloc[1:].to_numpy().argmin())]
    (OUT_DIR / "best_params.json").write_text(json.dumps(BEST, indent=2))
    print(f"tuned in {time.time() - t0:.0f}s | best {BEST}")
    display(GRID.round(4))
else:
    BEST = None
    print("the tune block is not in BLOCKS: nothing to tune")
TUNE_SECONDS = time.time() - t0

## 9. Refit the best setting for the test folds and for the frozen replay
**What happens here:** the best setting is trained again from scratch,
- once on the labels before `TEST_START` (2026-08-01) to forecast the **test** folds, and
- once on the labels before `CUTOFF` (2026-09-01) to forecast the **replay**, where the model then stays frozen
  (it does not learn from anything after the cut: this is how it would run "live").

The final weights are saved as a folder that `TimestralForecaster.from_pretrained(folder)` can reload (the LoRA
correction merged into the base weights). For the replay model that reload is **checked**: the reloaded model must
give the same forecasts as the one in memory.

Finally the result files are written in the exact layout of notebooks 05 and 05b so that notebook 06 can read them:
`test.parquet` and `replay.parquet`, with the columns
`model, scope, device_id, target, [fold,] origin, horizon, ts_hour, actual, point, q0.1 ... q0.9`.

In [ ]:
# =====================================================================================================
# WHAT        Refits BEST for the test and the replay periods, forecasts them, saves the final models and writes
#             test.parquet and replay.parquet.
# NEEDS       BEST, WINS, T_TEST, CUT, finetune, forecast_windows, cached_forecast.
# MAKES       FINAL {"test": table, "replay": table}, EVAL_SECONDS; files out/<run>/test.parquet, replay.parquet,
#             models/test/ and models/replay/ (if SAVE_MERGED).
# CONDITIONS  a block is skipped if it is not in BLOCKS, has no windows, or BEST is None. STOPS if the reloaded
#             replay model differs from the in-memory one by a relative 1e-3 or more.
# =====================================================================================================
from safetensors.torch import save_file


def save_merged(fc, folder):
    """Write config.json + model.safetensors with the LoRA correction merged into the base weights.

    The folder can be loaded with TimestralForecaster.from_pretrained(str(folder)) like any Timestral model.
    """
    folder.mkdir(parents=True, exist_ok=True)
    state = {}
    lora_layers = set()                                    # names of the wrapped layers, e.g. "...attns.0.wQKV"
    for name_, mod in fc.model.named_modules():
        if isinstance(mod, LoRALinear):                    # a wrapped layer: weight = original + correction
            lora_layers.add(name_)
            state[f"{name_}.weight"] = mod.merged_weight().detach().cpu().contiguous().clone()
    for k, v in fc.model.state_dict().items():             # every other weight is copied as it is
        if ".base." in k or k.rsplit(".", 1)[0] in lora_layers:
            continue                                       # skip the original layer inside a LoRALinear (its weight
                                                           # is already merged above) and the A and B matrices
        state[k] = v.detach().cpu().contiguous().clone()
    save_file(state, str(folder / "model.safetensors"))
    shutil.copy(resolve_checkpoint(MODEL_ID) / "config.json", folder / "config.json")


t0, FINAL = time.time(), {}
for block, end in [("test", T_TEST), ("replay", CUT)]:    # (period, the cut: train on labels before it)
    W = WINS[WINS["block"] == block]
    if block not in BLOCKS or not len(W) or BEST is None:
        continue

    def make(block=block, end=end, W=W):
        fc, _ = finetune(end, **BEST, name=block)           # train the best setting on labels before the cut
        df = forecast_windows(fc, W, block)                 # forecast the period's windows
        if SAVE_MERGED:
            save_merged(fc, OUT_DIR / "models" / block)
            if block == "replay":                           # the saved folder must reproduce the in-memory model
                chk = W.head(4)
                c, k, _ = batch_inputs(list(zip(chk["device_id"], chk["target"])), chk["p"].tolist())
                re = TimestralForecaster.from_pretrained(str(OUT_DIR / "models" / block), device=DEVICE)
                a = re.forecast(c[:, None, :], HORIZON, known_covariates=k).quantiles.float().cpu()
                b = fc.forecast(c[:, None, :], HORIZON, known_covariates=k).quantiles.float().cpu()
                rel = ((a - b).abs().max() / b.abs().max().clamp_min(1e-9)).item()
                print(f"reload check: max relative difference {rel:.2e}")
                assert rel < 1e-3, "the saved model does not reproduce the trained one"
                del re
        del fc
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return df

    FINAL[block] = cached_forecast(f"final_{block}", make)
EVAL_SECONDS = time.time() - t0

# Output layout (identical to notebook 05): which columns, in which order, with which types.
COLS = ["model", "scope", "device_id", "target", "origin", "horizon", "ts_hour", "actual", "point", *QCOLS]
TYPES = {**dict.fromkeys(["model", "scope", "device_id", "target"], object),
         **dict.fromkeys(["fold", "origin", "ts_hour"], "datetime64[ns, UTC]"),
         **dict.fromkeys(["point", "actual", *QCOLS], "float64")}
for block, df in FINAL.items():
    # "pooled" = one model for all devices, as for Chronos-2
    out = df.assign(model=RUN_NAME, scope="pooled").astype(TYPES)
    if block == "test":
        out[[*COLS[:4], "fold", *COLS[4:]]].to_parquet(OUT_DIR / "test.parquet", index=False)
    else:
        out[COLS].to_parquet(OUT_DIR / "replay.parquet", index=False)
print(f"test + replay in {EVAL_SECONDS:.0f}s | files in {OUT_DIR}")

## 10. Sanity checks (declared rules)
These checks must all pass; if one fails the forecasts must not be used.
- **C1 finite:** no NaN or infinity in any forecast.
- **C2 no crossing:** `q0.1 <= q0.25 <= q0.5 <= q0.75 <= q0.9` on every row (a higher quantile is never below a lower one).
- **C3 non-negative:** no forecast below 0 (energy cannot be negative).
- **C4 complete:** every row has its actual, and no more windows than planned were forecast.
- **C5 no leakage:** every fit's training windows end before its cut. This is asserted inside `train_pool` (section 7),
  so it has already been enforced before this point.

In [ ]:
# =====================================================================================================
# WHAT        Checks the final forecasts against the rules C1-C4 and prints PASS or FAIL for each.
# NEEDS       FINAL, QCOLS, WINS.
# MAKES       the dict `checks`.
# CONDITIONS  STOPS the notebook (assert) if any check fails.
# =====================================================================================================
checks = {}
for block, df in {**FINAL}.items():
    Q = df[QCOLS].to_numpy()                          # all quantile forecasts of this block as one array
    n_win = (WINS["block"] == block).sum()            # how many windows we planned for this block
    checks[f"{block} C1 finite"] = bool(np.isfinite(Q).all())
    checks[f"{block} C2 no crossing"] = bool((np.diff(Q, axis=1) >= -1e-9).all())
    checks[f"{block} C3 non-negative"] = bool((Q >= 0).all())
    checks[f"{block} C4 complete"] = bool(df["actual"].notna().all() and
                                          df.groupby(["device_id", "target", "origin"]).ngroups <= n_win)
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values()), "a sanity check failed: do not use these forecasts"

## 11. First look (descriptive only, not the final comparison)
The proper comparison with every other model (Holm-corrected Diebold-Mariano tests) happens in notebook 06. Here we
only show: the scores of this model on the test and replay windows, the training loss curves, and a small
`run_info.json` that records everything needed to reproduce the run.

**Scores (units kWh per hour unless noted; for MAE, RMSE and pinball lower is better):**
- **MAE** = `mean |actual - point forecast|`.
- **RMSE** = `sqrt(mean (actual - point forecast)^2)`; punishes big misses more.
- **pinball** = the quantile loss of section 5, here averaged over our 5 levels (no scale normalisation).
- **cov50 / cov80** = share of actuals inside the 25-75% / 10-90% band. A well-calibrated band gives about 0.50 / 0.80;
  much higher means the bands are too wide (cautious), much lower means overconfident.

**The loss curve** shows the training loss (25-step average). It should fall quickly at first and then flatten; if it
keeps falling while the tuning score gets worse, the model is overfitting.

In [ ]:
# =====================================================================================================
# WHAT        Prints the scores per block and target, plots the training loss, writes run_info.json.
# NEEDS       FINAL, LOSS_LOG, GRID, BEST, QCOLS, QUANTILES, the timing variables.
# MAKES       SUMMARY (table), out/<run>/summary.csv, plot_loss.png, run_info.json.
# CONDITIONS  none (with no FINAL tables the scores part is simply skipped).
# =====================================================================================================
import matplotlib.pyplot as plt

QA = np.array(QUANTILES)      # the quantile levels as an array, for the pinball formula


def metrics(g):
    """The scores of a group of forecast rows (see the text above)."""
    y, err = g["actual"], g["point"] - g["actual"]
    diff = y.to_numpy()[:, None] - g[QCOLS].to_numpy()                             # actual - quantile forecast
    return pd.Series({"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()),
                      "pinball": np.maximum(QA * diff, (QA - 1) * diff).mean(),
                      "cov50": y.between(g["q0.25"], g["q0.75"]).mean(),
                      "cov80": y.between(g["q0.1"], g["q0.9"]).mean(), "rows": len(g)})


# every table in FINAL already has a "block" column ("test" or "replay"), so just stack them and group by it
SUMMARY = (pd.concat(list(FINAL.values()), ignore_index=True)
           .groupby(["block", "target"]).apply(metrics, include_groups=False)) if FINAL else pd.DataFrame()
if len(SUMMARY):
    SUMMARY.to_csv(OUT_DIR / "summary.csv")
    display(SUMMARY.round(4))

if LOSS_LOG:
    ll = pd.DataFrame(LOSS_LOG)
    fig, ax = plt.subplots(figsize=(9, 4))
    for name_, g in ll.groupby("name", sort=False):                 # one line per fit (tune0, tune1, ..., test, replay)
        ax.plot(g["step"], g["loss"].rolling(25, min_periods=1).mean(), label=name_)
    ax.set(xlabel="training step", ylabel="scaled pinball loss (25-step mean)", title="Training loss")
    ax.grid(alpha=.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "plot_loss.png", dpi=120)
    plt.show()

run_info = {"model_id": MODEL_ID, "device": DEVICE, "torch": torch.__version__, "params": PARAMS,
            "best": BEST, "tune_seconds": round(TUNE_SECONDS), "eval_seconds": round(EVAL_SECONDS),
            "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if DEVICE == "cuda" else None,
            "windows": WINS.groupby("block").size().to_dict()}
(OUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(json.dumps({k: run_info[k] for k in ("best", "tune_seconds", "eval_seconds", "peak_vram_gb")}, default=str))

## 12. Running on the GX10
**Why the GX10:** the laptop GPU (4 GB) is too small to fine-tune. The job runs on the GX10 inside the
`chronos-sandbox-nv` container, with the `/w/.venv-timestral` Python environment (it has `timestral`, `torch` with
CUDA and `papermill`). The container mounts `~/chronos-sandbox` as `/w` and is capped at 24 GB. Each job gets its own
folder holding this notebook, the three frozen data files, the executed copy and the log. **Nothing here pushes to
GitHub.**

**How to use it (from the laptop notebook):**
- `gx10_run(...)` copies the notebook and data to a job folder on the box and starts the run in the background.
  Keyword arguments override the parameter cell. A quick smoke test, for example:
  `gx10_run(RUN_NAME="smoke", DEVICES=["<one PV device id>"], FT_GRID=[[50, 1e-4]], BLOCKS=["tune"])`.
  The real run is just `gx10_run()`.
- `gx10_status()` shows running jobs, the last log lines and the free memory.
- `gx10_pull()` copies the results back to `runs/gx10/timestral-ft/` (without the inputs and the large merged models).

`GX10_MIN_FREE_GB = 15` is a safety guard: `gx10_run` refuses to start when the box has less memory available, because
the box freezes if memory runs out. The host address (`GX10_HOST`) is read from the environment or the repo `.env`.

In [ ]:
# =====================================================================================================
# WHAT        Defines three helper functions to start, watch and collect a job on the GX10. (Definitions only.)
# NEEDS       NB_DIR, DATA_DIR, PARAMS (this notebook's parameters) and the GX10_HOST setting.
# MAKES       gx10_run(), gx10_status(), gx10_pull().
# CONDITIONS  gx10_run stops if GX10_HOST is not set, or if the box has less than GX10_MIN_FREE_GB GB available.
#             These helpers are meant to be called from the laptop; on the box the cell only defines them.
# =====================================================================================================
import shlex
import subprocess

GX10_DIR = "chronos-sandbox"                 # on the box, relative to the home folder
GX10_CONTAINER = "chronos-sandbox-nv"        # the GPU container the job runs in
GX10_JOBS = "runs/timestral-ft/jobs"         # where job folders are created (inside GX10_DIR)
GX10_MIN_FREE_GB = 15                        # refuse to start with less available memory than this
NB_FILE = "05c_timestral_209M_finetuned.ipynb"   # this notebook's file name


def _gx10_host():
    """The box address "user@host": the shell environment wins, otherwise GX10_HOST from the repo .env."""
    from dotenv import dotenv_values
    repo = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())   # the repository root
    host = os.environ.get("GX10_HOST") or dotenv_values(repo / ".env").get("GX10_HOST")
    if not host:
        raise RuntimeError("set GX10_HOST (user@host) in the environment or the repo .env")
    return host.strip(), repo


def gx10_run(**overrides):
    """Copy this notebook and the frozen data to a job folder on the box and run it there in the GPU container.

    overrides: parameter names and values that replace the ones in the parameter cell, e.g. DEVICES=[...].
    The job is named after RUN_NAME, so give a smoke test its own RUN_NAME.
    """
    host, _ = _gx10_host()
    free = subprocess.run(["ssh", host, "free -g | awk '/Mem:/ {print $7}'"], check=True,
                          capture_output=True, text=True).stdout            # GB of memory available on the box
    assert int(free) >= GX10_MIN_FREE_GB, f"only {free.strip()} GB available on the GX10"
    p = {**PARAMS, **overrides}                                              # parameters of this job
    job = p["RUN_NAME"]
    jobdir = f"{GX10_JOBS}/{job}"
    subprocess.run(["ssh", host, f"mkdir -p {GX10_DIR}/{jobdir}/data"], check=True)
    subprocess.run(["rsync", "-az", str(NB_DIR / NB_FILE), f"{host}:{GX10_DIR}/{jobdir}/"], check=True)
    data = [str(DATA_DIR / f) for f in ["processed_hourly_2026-10-25.parquet", "device_quality_2026-10-25.csv",
                                         "SHA256SUMS"]]
    subprocess.run(["rsync", "-az", *data, f"{host}:{GX10_DIR}/{jobdir}/data/"], check=True)
    w = f"/w/{jobdir}"                                                       # the same folder as the container sees it
    remote = (f"cd {GX10_DIR}/{jobdir} || exit 1; "
              f"nohup docker exec -e HOME=/tmp -e HF_HOME=/w/hf-cache -e HF_HUB_OFFLINE=1 {GX10_CONTAINER} "
              f"/w/.venv-timestral/bin/papermill {w}/{NB_FILE} {w}/05c.out.ipynb --cwd {w} -y "
              f"{shlex.quote(json.dumps(p))} > log.txt 2>&1 < /dev/null &")   # runs in the background, log in log.txt
    subprocess.run(["ssh", host, remote], check=True)
    print(f"started job {job} on {host}")


def gx10_status():
    """Show the running jobs with the tail of their logs, the newest job folder, and the box's free memory."""
    host, _ = _gx10_host()
    remote = (f"cd {GX10_DIR}/{GX10_JOBS}; "
              f"for j in $(docker exec {GX10_CONTAINER} pgrep -af papermill | grep -o 'jobs/[^/]*' "
              "| sort -u | cut -d/ -f2); do echo \"running: $j\"; "
              "tail -c 700 $j/log.txt | tr '\\r' '\\n' | tail -n 4; done; "
              "echo latest: $(ls -td */ | head -1); free -g | head -2")
    print(subprocess.run(["ssh", host, remote], capture_output=True, text=True).stdout)


def gx10_pull():
    """Copy the jobs back to runs/gx10/timestral-ft/ (without the inputs and the large merged models)."""
    host, repo = _gx10_host()
    local = repo / "runs" / "gx10" / "timestral-ft"
    subprocess.run(["rsync", "-az", "--exclude", "data/", "--exclude", "models/",
                    f"{host}:{GX10_DIR}/runs/timestral-ft/", str(local)], check=True)
    print("pulled to", local)